In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv("D:\\New Csv folder\\train.csv\\train.csv")
df.sample(5)

,id,qid1,qid2,question1,question2,is_duplicate
247403,247403,360629,257581,What is the most nutrient-dense food?,What is the most nutritionally dense food?,0
58654,58654,6803,15731,How can I stop masturbating forever?,How can one stop masturbation?,1
46793,46793,83644,83645,Is the European Union destined to fail if it d...,Is the European Union a failed project?,0
348822,348822,477418,477419,I love her. What should I do?,"I don't love her, what should I do?",0
141734,141734,224925,36629,What are some things new employees should know...,What are some things new employees should know...,0


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 404290 entries, 0 to 404289
Data columns (total 6 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   id            404290 non-null  int64 
 1   qid1          404290 non-null  int64 
 2   qid2          404290 non-null  int64 
 3   question1     404289 non-null  object
 4   question2     404288 non-null  object
 5   is_duplicate  404290 non-null  int64 
dtypes: int64(4), object(2)
memory usage: 18.5+ MB


In [4]:
df.isnull().sum()

id              0
qid1            0
qid2            0
question1       1
question2       2
is_duplicate    0
dtype: int64

In [5]:
df = df.dropna(subset=['question1','question2'])

In [6]:
# duplicate row
df.duplicated().sum()

np.int64(0)

In [7]:
is_duplicate = df['is_duplicate'].value_counts()
total = df['is_duplicate'].count()

percentages = (is_duplicate / total * 100).round(2)

print("Distribution of duplicate vs non-duplicate questions:")
print(percentages)


Distribution of duplicate vs non-duplicate questions:
is_duplicate
0    63.08
1    36.92
Name: count, dtype: float64


In [8]:
# Repeated questions
qid = pd.Series(df['qid1'].tolist() + df['qid2'].tolist())

unique_questions = np.unique(qid).shape[0]
total_questions = qid.shape[0]
repeated_questions = qid.duplicated().sum()

print(f"Number of unique questions: {unique_questions}")
print(f"Number of total questions: {total_questions}")
print(f"Number of repeated questions: {repeated_questions}")

Number of unique questions: 537929
Number of total questions: 808574
Number of repeated questions: 270645


In [9]:
# removing html tags
import re
def stript(data):
    p = re.compile(r'<.*?>')
    return p.sub('',data)

df['question1'] = df['question1'].apply(stript)
df['question2'] = df['question2'].apply(stript)

In [10]:
# url removing
def url_remove(data):
    url_rm = re.compile(r'https?://\S+|www\.\S+')
    return url_rm.sub('',data)

df['question1'] = df['question1'].apply(url_remove)
df['question2'] = df['question2'].apply(url_remove)

In [11]:
# removing Punctuation
def remove_pun(data):
    punc_remove = re.compile(r"[^\w\s]")
    return punc_remove.sub('', str(data))

df['question1'] = df['question1'].apply(remove_pun)
df['question2'] = df['question2'].apply(remove_pun)

In [12]:
# Removing Stop words
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

def remove_stopwords(text):
    return " ".join([word for word in text.split() if word.lower() not in stop_words])

df['question1'] = df['question1'].apply(remove_stopwords)
df['question2'] = df['question2'].apply(remove_stopwords)

In [13]:
# stemmer 
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()

def stem_words(text):
    return " ".join([ps.stem(word) for word in text.split()])

df['question1'] = df['question1'].apply(stem_words)
df['question2'] = df['question2'].apply(stem_words)

In [14]:
df.head(3)

,id,qid1,qid2,question1,question2,is_duplicate
0,0,1,2,step step guid invest share market india,step step guid invest share market,0
1,1,3,4,stori kohinoor kohinoor diamond,would happen indian govern stole kohinoor kohi...,0
2,2,5,6,increas speed internet connect use vpn,internet speed increas hack dn,0


In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack

vector = TfidfVectorizer()

x1 = vector.fit_transform(df['question1'])
x2 = vector.transform(df['question2'])

X = hstack([x1, x2])
y = df['is_duplicate']

In [16]:
print(X.shape)
print(X[:5].toarray())

(404287, 125338)
[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]


In [17]:
X[1]

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 11 stored elements and shape (1, 125338)>

---
## ML Model Traning

In [18]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix

In [19]:
y = df['is_duplicate']
x_train, x_test, y_train, y_test = train_test_split(X,y,test_size=0.3,random_state=42)

In [20]:
# from sklearn.naive_bayes import MultinomialNB

# model = MultinomialNB()
# model.fit(x_train, y_train)

from xgboost import XGBClassifier
model = XGBClassifier()
model.fit(x_train, y_train)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [21]:
y_pred = model.predict(x_test)

print(confusion_matrix(y_test,y_pred))
print(accuracy_score(y_test,y_pred))

[[71476  5133]
 [26108 18570]]
0.7424208695078616
